# Namespaces and Decorators — Pythonic Thinking

Every name in a Python program — every variable, function, class, module — lives in a **namespace**: an object that maps names to the things they refer to. At module level you can think of a namespace as a dictionary; type `globals()` and you will literally see one. This notebook is about two related superpowers. First, *where Python looks* when you use a name — the **LEGB rule**, which decides whether `a` means your local variable, an enclosing function's variable, the module-level variable, or a built-in. Second, *what you can do with functions once you treat them as data* — passing them around, wrapping them, and finally the **decorator** syntax that lets you bolt extra behaviour onto a function without touching its body.

Why care? Because nearly every "why is my variable not updating?!" bug is a namespace bug: a function that assigns to a name silently creates a *local* variable instead of touching the global one. And nearly every framework you will meet — Flask, Django, pytest, click, FastAPI — is built on decorators, so reading `@app.route` or `@app.get` should feel ordinary, not mysterious.

The roadmap for this notebook:

1. **Namespaces and the LEGB rule** — the four scopes (built-in, global, enclosing, local), searched inside-out, with `NameError` as the final outcome.
2. **Local vs global in practice** — shadowing, read-only fallback, assignment surprises, and the `global` keyword for deliberate edits.
3. **The built-in scope** — the `builtins` module, and why redefining `max` (or `print`) is legal but dangerous.
4. **Enclosing scope and `nonlocal`** — nested functions reaching outward, and how `nonlocal` differs from `global`.
5. **Functions as first-class objects** — the prerequisite that makes decorators possible at all.
6. **Decorators** — the wrapper pattern by hand, the `@` syntax as shorthand, a genuinely useful `@timer`, the classic pitfalls, and finally **decorators with arguments** built from three nested functions.

Two definitions to carry forward. A **scope** is the region where a name is directly visible; a **closure** is a function plus the enclosing variables it remembers; a **decorator** is a callable that takes a function and returns a (usually enhanced) function. By the end, both halves of the notebook — name lookup and function wrapping — should feel like two views of the same idea: Python lets you intercept things, names and calls alike, at well-defined choke points.

# 1. Scope and the LEGB rule

## 1.1 Python searches inside-out

A **scope** is the textual region of a program where a namespace is directly accessible. Python keeps several namespaces alive at once and resolves every bare name by walking outward through them in a fixed order — the famous **LEGB** rule:

| Order | Scope | Where it lives | Typical names |
| --- | --- | --- | --- |
| 1 | **L**ocal | The executing function's own frame | parameters, locally assigned names |
| 2 | **E**nclosing | Frames of any *outer* functions | variables of an enclosing `def` |
| 3 | **G**lobal | The current module's top level | module-level variables and functions |
| 4 | **B**uilt-in | The `builtins` module, always present | `print`, `len`, `max`, `True`, `Exception` |

The search stops at the first hit. If all four miss, Python raises `NameError: name 'x' is not defined` — "not defined" really meaning *"not found in any accessible scope"*.

Two corollaries drive most scope bugs. First, **reading** a name you have not defined locally is fine when an outer scope has it — the lookup simply falls through. Second, **assigning** a name inside a function always targets the *local* scope unless you say otherwise: `a = 3` inside `temp()` creates a brand-new local `a` and leaves the global one untouched. Even `a += 1` counts as an assignment, so Python marks `a` as local for the whole function — which is why that pattern fails with `UnboundLocalError` unless a local value already exists.

Shadowing is silent. The cells below walk each combination one at a time: local and global used together, the same name in both scopes, read-only access to a global, attempts to edit a global from inside a function, and finally the `global` keyword that makes the edit intentional.

In [1]:
# local and global
# global var
a = 2

def temp():
  # local var
  b = 3
  print(b)

temp()
print(a)

3
2


In [2]:
# local and global -> same name
a = 2

def temp():
  # local var
  a = 3
  print(b)

temp()
print(a)

NameError: name 'b' is not defined

In [ ]:
# local and global -> local does not have but global has
a = 2

def temp():
  # local var
  print(a)

temp()
print(a)


In [ ]:
# local and global -> editing global
a = 2

def temp():
  # local var
  a += 1
  print(a)

temp()
print(a)

In [3]:
a = 2

def temp():
  # local var
  global a
  a += 1
  print(a)

temp()
print(a)

3
3


In [4]:
# local and global -> global created inside local
def temp():
  # local var
  global a
  a = 1
  print(a)

temp()
print(a)

1
1


In [5]:
# local and global -> function parameter is local
def temp(z):
  # local var
  print(z)

a = 5
temp(5)
print(a)
print(z)

5
5


NameError: name 'z' is not defined

## 1.2 The built-in namespace

At the bottom of the LEGB ladder sits a namespace you did not create and cannot erase: **built-ins**. Every name Python provides out of the box — `print`, `len`, `max`, `min`, `range`, `sum`, `bool`, `dict`, `Exception`, all the types and error classes — is stored in the `builtins` module and is visible from anywhere without an import. The next cell proves it: `import builtins` followed by `dir(builtins)` dumps the entire catalogue into the output, and the tiny placeholder cells that follow mark the spot for your own exploration of that list.

Built-ins behave like any other name, with one crucial tweak in the search order: they are checked **last**. That gives you a legal but hazardous power — you can redefine them. The following demonstration rebinds the name `max` to a zero-argument function that prints `'hello'`. From that line onward, the name `max` in the global namespace shadows the built-in `max` for everything that resolves the name afterwards, so a later call like `max([1, 2, 3])` fails: the name now points at *your* function, which accepts no arguments.

The lesson is about respect for the LEGB ladder. Defining a variable or function whose name collides with a built-in — `list = ...`, `def print():`, `def max():` — silently hides the original for every look-up after it, and the hidden version is not restored until the session restarts or the name is reassigned. The conventions of the community — `type_`, `max_value`, or simply honest longer names — exist precisely to avoid squatting on shared public infrastructure: built-ins are always readable when you leave them alone, and quietly broken when you do not.

In [6]:
# built-in scope
import builtins
print(dir(builtins))

['ArithmeticError', 'AssertionError', 'AttributeError', 'BaseException', 'BaseExceptionGroup', 'BlockingIOError', 'BrokenPipeError', 'BufferError', 'BytesWarning', 'ChildProcessError', 'ConnectionAbortedError', 'ConnectionError', 'ConnectionRefusedError', 'ConnectionResetError', 'DeprecationWarning', 'EOFError', 'Ellipsis', 'EncodingWarning', 'EnvironmentError', 'Exception', 'ExceptionGroup', 'False', 'FileExistsError', 'FileNotFoundError', 'FloatingPointError', 'FutureWarning', 'GeneratorExit', 'IOError', 'ImportError', 'ImportWarning', 'IndentationError', 'IndexError', 'InterruptedError', 'IsADirectoryError', 'KeyError', 'KeyboardInterrupt', 'LookupError', 'MemoryError', 'ModuleNotFoundError', 'NameError', 'None', 'NotADirectoryError', 'NotImplemented', 'NotImplementedError', 'OSError', 'OverflowError', 'PendingDeprecationWarning', 'PermissionError', 'ProcessLookupError', 'PythonFinalizationError', 'RecursionError', 'ReferenceError', 'ResourceWarning', 'RuntimeError', 'RuntimeWarning

In [7]:
# how to see all the built-ins

In [8]:
# renaming built-ins
L = [1,2,3]
print(max(L))
def max():
  print('hello')

print(max(L))

3


TypeError: max() takes 0 positional arguments but 1 was given

## 1.3 Enclosing scope and `nonlocal`

LEGB's **E** only exists when functions nest. In the next cell, `inner` is defined inside `outer`; when `inner` executes, it may read names from its own frame, then `outer`'s frame, then the module, then built-ins. That middle layer — the enclosing frame — is what makes **closures** possible: a function that keeps access to variables from the environment where it was born, even after that environment's function has returned.

Watch the first demo closely. `inner` prints `a`, yet neither `inner` nor `outer` defines `a` in that cell. The lookup sails past the inner local scope (nothing there), past the enclosing local scope (`outer` has no `a` either), and lands in the **global** scope, where the name `a` is still sitting with the value `5` left over from an earlier cell — which is why the stored output prints `5` before `outer function` and `main program`. In a clean session with no leftover global `a`, the same code would raise `NameError`. That is a healthy reminder that LEGB is a *fallback chain*, not a guarantee.

Then comes the `nonlocal` demo, where `inner` wants to *modify* `outer`'s `a`. A plain assignment would make `a` local to `inner` (shadowing), and `global a` would jump all the way to module level — both wrong targets. `nonlocal a` is the precise tool: "the `a` I mean lives in the nearest enclosing scope." With it, `a += 1` inside `inner` updates `outer`'s `a`, so both prints show `2`.

Rule of thumb: `global` targets the *module*; `nonlocal` targets the nearest enclosing *function*; and doing neither means a name you assign belongs to your *own* frame. The decorators in the second half of this notebook lean on exactly this enclosing-scope machinery.

In [9]:
# Enclosing scope
def outer():
  def inner():
    print(a)
  inner()
  print('outer function')


outer()
print('main program')

5
outer function
main program


In [10]:
# nonlocal keyword
def outer():
  a = 1
  def inner():
    nonlocal a
    a += 1
    print('inner',a)
  inner()
  print('outer',a)


outer()
print('main program')

inner 2
outer 2
main program


In [11]:
# Summary

# 2. Decorators

## 2.1 Bolting behaviour onto a function from the outside

A **decorator** is a function that takes a function as input, wraps it with extra behaviour, and returns a new function. Nothing in the original function's source changes — the enhancement arrives from the outside, like slipping a book into a gift wrap without rewriting a single page.

```python
def my_decorator(func):      # accepts a function
    def wrapper():           # defines the enhanced version
        print(...)           # extra behaviour before
        func()               # the original call
        print(...)           # extra behaviour after
    return wrapper           # hands back the enhanced version
```

This is only possible because of a feature you will prove in the very next cell: **functions are first-class citizens** — ordinary objects that can be stored in variables, passed as arguments and returned from other functions.

Python also ships a family of **built-in decorators** you will meet in object-oriented code: `@staticmethod` and `@classmethod` change how a method receives its first argument, `@property` turns a method into attribute-style access, and `@abstractmethod` marks a method that subclasses must implement. Everything else — timing, logging, validation, caching, web routing — is built by programmers as **user-defined decorators** shaped to their own needs.

Meanwhile, the `@` sign above a definition in later cells is not a new language feature at all: it is shorthand for a reassignment that executes the moment the `def` statement finishes. Understanding that one-line reassignment is ninety percent of the decorator story. The remaining ten percent is knowing that a robust wrapper forwards `*args, **kwargs`, returns the result, and preserves the original function's name and docstring.

## 2.2 First-class functions: the prerequisite

Before decorating anything, confirm the raw material. In Python, a function is a normal object: it has a name, lives in a namespace, can be stored in a container, and — the ingredient decorators need — can be **passed as an argument** and **returned as a result**.

The cell right after this one demonstrates the first half in the smallest possible frame:

```python
def modify(func, num):
    return func(num)

modify(square, 2)   # 4
```

`square` arrives as `func`, a plain parameter; calling `func(num)` is an ordinary call through a variable. If functions were compile-time blueprints you could not hand around, decorators would be impossible. Because functions are values, functions that *accept* functions are just utility functions — and functions that *return* functions are factories.

Three capabilities to tick off; each appears later in this notebook:

1. **Assign** a function to a name: `a = my_decorator(hello)`.
2. **Pass** a function into another: `modify(square, 2)`, and every decorator application.
3. **Return** a function from another: `my_decorator` returns `wrapper`; `sanity_check(int)` returns `outer_wrapper`.

A closure adds a fourth: the returned function can keep remembering variables from where it was created — which is precisely how `wrapper` holds onto `func` after `my_decorator` has already returned.

In [12]:
# Python are 1st class function

def modify(func,num):
  return func(num)

def square(num):
  return num**2

modify(square,2)

4

## 2.3 The wrapper pattern, by hand

Here is decoration with the training wheels still on — no `@` yet, just functions being honest about what they are.

`my_decorator(func)` defines an inner function called `wrapper` that does three things: print a line of asterisks, call `func()`, print another line of asterisks. Then it **returns** `wrapper` itself — not the result of calling it. Meanwhile `wrapper` keeps a live reference to the parameter `func`; even though `my_decorator(hello)` has finished executing, `wrapper` can still call `hello`. A function that remembers names from its enclosing scope after that scope has returned is called a **closure** — and it is the engine under every decorator.

Usage in the demo cells is explicit:

```python
a = my_decorator(hello)   # a IS wrapper, with func bound to hello
a()                       # asterisks, 'hello', asterisks

b = my_decorator(display) # same wrapper shape, different func
b()                       # asterisks, 'hello nitish', asterisks
```

Notice that `hello` and `display` themselves were never edited — their bodies still print exactly what they always printed. Decoration is *outside-in*: behaviour is added by wrapping, never by modifying. Two details to file away: the wrapper defines no `return`, so decorated calls currently produce `None` (worth remembering when we reach the `timer` decorator), and the same `my_decorator` function formalises around two different functions, which is exactly the reusability that makes decorators a language feature rather than a one-off trick.

In [13]:
# simple example

def my_decorator(func):
  def wrapper():
    print('***********************')
    func()
    print('***********************')
  return wrapper

def hello():
  print('hello')

def display():
  print('hello nitish')

a = my_decorator(hello)
a()

b = my_decorator(display)
b()

***********************
hello
***********************
***********************
hello nitish
***********************


In [14]:
# more functions

In [15]:
# how this works -> closure?

In [16]:
# python tutor

## 2.4 The `@` syntax is a one-line reassignment

Manual decoration works, but writing `hello = my_decorator(hello)` under every function gets old fast. Python's answer is the `@` line:

```python
@my_decorator
def hello():
    print('hello')
```

This is **exactly equivalent** to defining `hello` normally and immediately running `hello = my_decorator(hello)`. Nothing else changes: the name `hello` ends up bound to the wrapper, so the later bare call `hello()` executes the decorated version, printing the asterisk fences just as in the hand-written demo.

Three details worth memorising:

- **Timing.** The decorator runs once, at *definition* time — right after the `def` line executes while the cell runs. Every later call to the function just invokes the already-wrapped result.
- **Position.** `@decorator` must sit directly above a `def` (or `class`). Stack them for composition; the bottom decorator wraps first.
- **Naming honesty.** After decoration, `hello.__name__` reports `'wrapper'`, not `'hello'`, unless the wrapper takes protective measures. The name you see in tracebacks, `help()` output and logs changes — a small identity crisis with real debugging costs, flagged again in two sections' time.

In [17]:
# Better syntax?
# simple example

def my_decorator(func):
  def wrapper():
    print('***********************')
    func()
    print('***********************')
  return wrapper

@my_decorator
def hello():
  print('hello')

hello()

***********************
hello
***********************


## 2.5 A decorator that earns its keep: `@timer`

Time to decorate something useful. The `timer` decorator wraps a function so that every call reports how long it ran:

```python
def timer(func):
    def wrapper(*args):
        start = time.time()
        func(*args)
        print('time taken by', func.__name__, time.time() - start, 'secs')
    return wrapper
```

Read the three moves. `wrapper(*args)` accepts **any number of positional arguments** — that single `*args` is what lets one decorator serve a zero-argument `hello()`, a one-argument `square(2)` and a two-argument `power(2, 3)` without knowing their signatures in advance. (This version does not forward `**kwargs`, so keyword-only calls would fail — a gap the robust template closes as well.) It then records `time.time()`, calls the original function, and prints the elapsed seconds alongside `func.__name__`, which still refers to the *original* function because the wrapper closed over it.

Applied via `@timer`, the three demo calls show the value immediately: `hello` reports roughly its `time.sleep(2)`, `square` reports about a second of sleeping, and `power` reports a near-zero runtime for `2**3 = 8`. One decorator, three unrelated functions, zero edits to their bodies — decoration as a drop-in measurement harness, reusable anywhere you want cross-cutting behaviour such as timing, logging, retries or access checks without repeating yourself.

In [18]:
# anything meaningful?
import time

def timer(func):
  def wrapper(*args):
    start = time.time()
    func(*args)
    print('time taken by',func.__name__,time.time()-start,'secs')
  return wrapper

@timer
def hello():
  print('hello wolrd')
  time.sleep(2)

@timer
def square(num):
  time.sleep(1)
  print(num**2)

@timer
def power(a,b):
  print(a**b)

hello()
square(2)
power(2,3)


hello wolrd
time taken by hello 2.0006842613220215 secs
4
time taken by square 1.0013370513916016 secs
8
time taken by power 2.5033950805664062e-05 secs


## 2.6 The two classic wrapper pitfalls

The next cell is flagged as a problem checkpoint in the notebook — let us name the problems it points at. Both are visible in the `timer` wrapper we just read.

**1. Swallowed return values.** The wrapper calls `func(*args)` but never returns the result. For `hello()` that is harmless — a print-only function was returning `None` anyway. For `square(2)` it is a genuine bug: the decorated function now evaluates to `None`, so `result = square(2) * 10` would blow up even though the undecorated version worked. The fix is one keyword: `return func(*args)`.

**2. Lost identity.** After `@timer` is applied, `hello.__name__` reports `'wrapper'`, because the name lookup finds the wrapper function, not the original. Tracebacks, logs, `help()` output and any framework that inspects function names now see the wrong label. The standard remedy is `functools.wraps` applied *inside* the decorator; it copies `__name__`, `__doc__` and friends from the original function onto the wrapper.

Neither pitfall is exotic — both appear constantly in real code, which is why the community template for a decorator looks like this: accept `*args, **kwargs`; do the pre-work; `result = func(*args, **kwargs)`; do the post-work; `return result` — with `@functools.wraps(func)` glued on top of the wrapper. Keep that five-line skeleton in mind and you can read almost any decorator you meet in production.

In [19]:
# A big problem

# 3. Decorators with arguments

## 3.1 Three nested functions: factory, decorator, wrapper

So far our decorators were applied as `@timer`. But what about `@sanity_check(int)` — a decorator that *itself* takes arguments? The trouble: Python evaluates `sanity_check(int)` **first**, and whatever it returns is what gets applied to the function below it. So `sanity_check(int)` must return... a decorator. And that decorator must return a wrapper. Hence three levels:

```python
def sanity_check(data_type):      # level 1: called with the argument
    def outer_wrapper(func):      # level 2: a normal decorator
        def inner_wrapper(*args): # level 3: the actual wrapper
            if type(*args) == data_type:
                func(*args)
            else:
                raise TypeError(...)
        return inner_wrapper
    return outer_wrapper
```

Trace `@sanity_check(int)` above a function: `sanity_check(int)` runs and returns `outer_wrapper`; Python then executes `square = outer_wrapper(square)`; `outer_wrapper` returns `inner_wrapper`; `square` ends up bound to that. On every call, `inner_wrapper` checks the first argument's type against `data_type` and either forwards the call or raises a `TypeError` with a custom message — a guard rail fitted around the function without touching its body. As written, the check assumes a single argument; a robust version would inspect all of `args`.

The cells that follow show the two halves of this idea. One sketches the intended usage with a `@checkdt(int)` style call, and the next holds the full `sanity_check` implementation — same three-level shape, different label. Two base configurations are registered from one recipe: `square` restricted to `int`, and `greet` restricted to `str`. Rule of thumb for the level count: **one extra argument in the `@` line means one extra nested function** — arguments in, decorator out, wrapper last.

In [20]:
# One last example -> decorators with arguments


In [21]:
@checkdt(int)
def square(num):
  print(num**2)

NameError: name 'checkdt' is not defined

In [23]:
def sanity_check(data_type):
  def outer_wrapper(func):
    def inner_wrapper(*args):
      if type(*args) == data_type:
        func(*args)
      else:
        raise TypeError('Ye datatype nai chalega')
    return inner_wrapper
  return outer_wrapper

@sanity_check(int)
def square(num):
  print(num**2)

@sanity_check(str)
def greet(name):
  print('hello',name)

square(2)

4


## Summary

Namespaces are the dictionaries that bind names to objects, and Python resolves every name through the **LEGB** chain — local, enclosing, global, built-in — stopping at the first hit and raising `NameError` when all four miss. Assignments always create or update the *local* name unless `global` (module level) or `nonlocal` (nearest enclosing function) says otherwise; built-ins sit at the bottom of the ladder and can be accidentally shadowed. Functions, being first-class objects, can be passed and returned — which makes a **decorator** possible: a wrapper-producing function that enhances another function without editing it. The `@name` line is shorthand for a reassignment that happens at definition time; robust wrappers forward `*args, **kwargs`, return the result, and preserve metadata with `functools.wraps`. Giving a decorator its own arguments simply adds a nesting level — factory, decorator, wrapper — exactly as `sanity_check(data_type)` demonstrates.